In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy import signal

# ==========================================
# 1. CONFIGURACIÓN DE ONDAS Y PROTOCOLOS DINÁMICOS
# ==========================================
configuraciones_ondas = {
    'Ascenso Dinámico: Beta (20 Hz) ➔ Gamma (40 Hz)': {
        'tipo_progreso': 'glide',
        'freq_inicial': 20.0,
        'freq_final': 40.0,
        'color': '#e67e22',
        'induccion_min': '8-12 min',
        'sistema': 'Corteza Prefrontal hacia Redes de Alta Sincronía Cortical (Binding Perceptual).',
        'susurro_impacto': 'El deslizamiento suave acompañado de susurros rítmicos evita la fatiga sináptica al acelerar de la lógica a la lucidez.',
        'objetivo': 'Tránsito fluido desde el análisis lógico profundo hacia el estado de hiperlucidez e integración "Eureka".',
        'ventajas': 'Permite resolver problemas técnicos complejos y alcanzar estados de alta creatividad sin sobrecargar el sistema nervioso.',
        'tipo_audio_sugerido': 'Glide armónico ascendente con texturas de cuarzo/citrino y sobretonos cristalinos limpios.'
    },
    'Theta (6 Hz)': {
        'tipo_progreso': 'estatico', 'freq': 6.0, 'color': '#8e44ad',
        'induccion_min': '7-10 min',
        'sistema': 'Sistema Límbico (Hipocampo) y Sistema Parasimpático Vagal.',
        'susurro_impacto': 'Susurro rítmico lento que simula sueño profundo y desactiva la amígdala.',
        'objetivo': 'Creatividad, acceso a memoria implícita e imágenes hipnagógicas.',
        'ventajas': 'Facilita la resolución de problemas abstractos y reduce la fatiga mental profunda.',
        'tipo_audio_sugerido': 'Pulsos binaurales suaves sobre colchón de ruido rosa con susurros.'
    },
    'Alfa (10 Hz)': {
        'tipo_progreso': 'estatico', 'freq': 10.0, 'color': '#16a085',
        'induccion_min': '5-10 min',
        'sistema': 'Corteza Occipito-Parietal y Redes de Atención Sostenida.',
        'susurro_impacto': 'Ancla sensorial que reduce la hipervigilancia visual y relaja los músculos perioculares.',
        'objetivo': 'Concentración sostenida, lectura comprensiva y estado de Flow operativo.',
        'ventajas': 'Disminuye la ansiedad por sobrecarga y mejora la fluidez en tareas de enfoque.',
        'tipo_audio_sugerido': 'Ondas isocrónicas moderadas con brisa o susurro ambiental continuo.'
    },
    'Gamma (40 Hz Estática)': {
        'tipo_progreso': 'estatico', 'freq': 40.0, 'color': '#c0392b',
        'induccion_min': '10-15 min',
        'sistema': 'Redes de Alta Conectividad Cortical (Interhemisférica).',
        'susurro_impacto': 'Micro-transiciones rápidas que obligan a integrar información de alto nivel.',
        'objetivo': 'Procesamiento de información avanzada y momentos de máxima claridad.',
        'ventajas': 'Favorece la síntesis conceptual rápida.',
        'tipo_audio_sugerido': 'Espectro rico (diente de sierra) con armónicos agudos.'
    },
    'Resonancia (111 Hz)': {
        'tipo_progreso': 'estatico', 'freq': 111.0, 'color': '#2980b9',
        'induccion_min': '7-12 min',
        'sistema': 'Sistema Somatosensorial, Estructura Esqueleto-Craneal y Nervio Vago.',
        'susurro_impacto': 'Resonancia ósea directa combinada con un susurro grave de campo seguro.',
        'objetivo': 'Relajación somática profunda y liberación de tensión física.',
        'ventajas': 'Alivia la tensión muscular y promueve anclaje físico unificado.',
        'tipo_audio_sugerido': 'Tonos de baja frecuencia con fuerte resonancia armónica.'
    }
}

opciones_ondas = {
    'Sinusoidal (Tono Puro - k=1)': 'sin',
    'Cuadrada (Armónicos Impares)': 'square',
    'Diente de Sierra (Espectro Completo)': 'sawtooth'
}

combinaciones_acusticas = {
    'Cuencos de Cuarzo/Citrino (Claridad & Sobretonos Agudos)': {
        'desc': 'Armónicos cristalinos puros. Ideal para potenciar bandas Gamma y claridad mental.',
        'mod_textura': 1.25
    },
    'Cuencos de Cobre (Resonancia Somática Profunda)': {
        'desc': 'Vibraciones graves de baja frecuencia. Anclaje físico directo y tejido óseo.',
        'mod_textura': 0.85
    },
    'Hibrido: Cuarzo/Citrino + Susurro Respiratorio (Vagal)': {
        'desc': 'Fusión sinérgica: claridad mental elevada combinada con relajación parasimpática.',
        'mod_textura': 1.2
    }
}

# ==========================================
# 2. MOTOR DE SIMULACIÓN TMNE v2.5 CON GLIDE Y N VARIABLE
# ==========================================
def ejecutar_simulacion_glide(config_data, tipo_onda, factor_textura, n_neurons):
    C_m = 1.0
    g_L = 0.1
    V_L = -70.0
    V_th = -50.0
    V_reset = -75.0
    E_Ca = 120.0
    g_CaT = 0.8

    dt = 0.05
    T_total = 500.0
    time = np.arange(0, T_total, dt)
    A_field = 0.6 * factor_textura
    D_noise = 2.5

    np.random.seed(42)

    # Definir la trayectoria de frecuencia (Estática o Glide)
    if config_data['tipo_progreso'] == 'glide':
        f_ini = config_data['freq_inicial']
        f_fin = config_data['freq_final']
        drift_frecuencia = np.piecewise(time,
                                        [time <= 350, time > 350],
                                        [lambda t: f_ini + (f_fin - f_ini) * (t / 350.0), f_fin])
    else:
        freq_base = config_data['freq']
        drift_frecuencia = freq_base + 0.5 * np.sin(2 * np.pi * 0.002 * time)

    fase_acumulada = 2 * np.pi * np.cumsum(drift_frecuencia / 1000.0 * dt)
    jitter_fase = 0.05 * np.random.normal(0, 1, len(time))
    fase_total = fase_acumulada + jitter_fase

    if tipo_onda == 'square':
        wave_signal_raw = signal.square(fase_total)
    elif tipo_onda == 'sawtooth':
        wave_signal_raw = signal.sawtooth(fase_total)
    else:
        wave_signal_raw = np.sin(fase_total)

    i_ext_vals = A_field * wave_signal_raw

    n_direct = int(0.25 * n_neurons)
    direct_indices = np.random.choice(n_neurons, n_direct, replace=False)
    mask_recurrent = ~np.isin(np.arange(n_neurons), direct_indices)

    m_inf = lambda v: 1.0 / (1.0 + np.exp(-(v + 63.0) / 7.8))
    h_inf = lambda v: 1.0 / (1.0 + np.exp((v + 84.0) / 5.2))

    V = np.ones(n_neurons) * V_L + np.random.normal(0, 1.0, n_neurons)
    h = h_inf(V)

    fptd_times = []
    actividad_global = np.zeros(len(time))

    J_syn = 2.2
    prev_activity_fraction = 0.0

    for step, t in enumerate(time):
        I_ext_vector = np.zeros(n_neurons)
        I_ext_vector[direct_indices] = i_ext_vals[step]
        I_ext_vector[mask_recurrent] = J_syn * prev_activity_fraction

        xi = np.random.normal(0, np.sqrt(2 * D_noise * dt), n_neurons)

        I_CaT = g_CaT * (m_inf(V)**2) * h * (V - E_Ca)
        I_L = g_L * (V - V_L)

        dV = (-I_L - I_CaT + I_ext_vector) * (dt / C_m) + xi
        V += dV
        h += (h_inf(V) - h) / 10.0 * dt

        spiked_mask = V >= V_th
        if np.any(spiked_mask):
            fptd_times.extend([t] * np.sum(spiked_mask))
            V[spiked_mask] = V_reset
            h[spiked_mask] = h_inf(V_reset)

        actividad_global[step] = np.sum(spiked_mask)
        prev_activity_fraction = np.sum(spiked_mask) / n_neurons

    return time, actividad_global, fptd_times, i_ext_vals, drift_frecuencia

# ==========================================
# 3. INTERFAZ Y VISUALIZACIÓN DINÁMICA
# ==========================================

selector_onda = widgets.Dropdown(
    options=list(configuraciones_ondas.keys()),
    value='Ascenso Dinámico: Beta (20 Hz) ➔ Gamma (40 Hz)',
    description='Protocolo:',
    style={'description_width': 'initial'}
)

dropdown_forma_onda = widgets.Dropdown(
    options=list(opciones_ondas.keys()),
    value='Sinusoidal (Tono Puro - k=1)',
    description='Armónicos:',
    style={'description_width': 'initial'}
)

dropdown_combinacion = widgets.Dropdown(
    options=list(combinaciones_acusticas.keys()),
    value='Hibrido: Cuarzo/Citrino + Susurro Respiratorio (Vagal)',
    description='Textura Acústica:',
    style={'description_width': 'initial'}
)

dropdown_neuronas = widgets.Dropdown(
    options={
        '1,000 Neuronas (Red Reducida)': 1000,
        '2,500 Neuronas (Red Media)': 2500,
        '5,000 Neuronas (Estándar TMNE)': 5000,
        '10,000 Neuronas (Alta Densidad)': 10000
    },
    value=5000,
    description='Nº Neuronas (N):',
    style={'description_width': 'initial'}
)

output = widgets.Output()
output_guia = widgets.Output()

def actualizar_todo(change):
    with output:
        clear_output(wait=True)
        clave_protocolo = selector_onda.value
        nombre_onda_amigable = dropdown_forma_onda.value
        tipo_onda = opciones_ondas[nombre_onda_amigable]
        nombre_combinacion = dropdown_combinacion.value
        n_neurons = dropdown_neuronas.value

        datos_protocolo = configuraciones_ondas[clave_protocolo]
        info_combinacion = combinaciones_acusticas[nombre_combinacion]

        t, act, fptd, ie, perfil_freq = ejecutar_simulacion_glide(
            datos_protocolo, tipo_onda, info_combinacion['mod_textura'], n_neurons
        )

        if fptd:
            sorted_fptd = np.sort(fptd)
            latencia_activacion = np.median(sorted_fptd)
            factor_armonico = 1.0 if tipo_onda == 'sin' else (0.8 if tipo_onda == 'square' else 0.65)

            limpio = datos_protocolo['induccion_min'].replace(' min', '')
            exposicion_analitica_min = float(limpio.split('-')[0]) * factor_armonico
            exposicion_analitica_max = float(limpio.split('-')[1]) * factor_armonico
        else:
            latencia_activacion = 250.0
            exposicion_analitica_min, exposicion_analitica_max = 5.0, 10.0

        fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(11, 17.5))
        fig.subplots_adjust(hspace=0.6)

        fig.suptitle(f"TMNE v2.5: Protocolo Acústico Avanzado [{clave_protocolo}] | N={n_neurons}\n",
                     fontsize=12, fontweight='bold', color='#2c3e50', y=0.99)

        fig.text(0.5, 0.955, f"Latencia Umbral (50%): {latencia_activacion:.1f} ms  |  Ventana Óptima: {exposicion_analitica_min:.1f} - {exposicion_analitica_max:.1f} min",
                 fontsize=10, ha='center', color='#c0392b', fontweight='bold')

        # --- Gráfico 1 ---
        ax1.plot(t, perfil_freq, color=datos_protocolo['color'], linewidth=2.5)
        ax1.set_ylabel("Freq (Hz)", fontweight='bold', color='#2c3e50')
        ax1.set_title("1. Trayectoria de Frecuencia (Desplazamiento Dinámico / Glide)", fontsize=10, fontweight='bold', color='#2c3e50')
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.set_xlim(0, 500)

        # --- Gráfico 2 ---
        ax2.plot(t, act, color=datos_protocolo['color'], linewidth=2)
        ax2.fill_between(t, act, color=datos_protocolo['color'], alpha=0.25)
        ax2.set_ylabel("Neuronas Activas", fontweight='bold', color='#2c3e50')
        ax2.set_title(f"2. Respuesta en Cascada de la Red (N={n_neurons})", fontsize=10, fontweight='bold', color='#2c3e50')
        ax2.grid(True, linestyle=':', alpha=0.6)
        ax2.set_xlim(0, 500)

        # --- Gráfico 3 ---
        idx_inicio = t <= 75
        idx_final = t >= 425
        ax3.plot(t[idx_inicio], act[idx_inicio], color='#2c3e50', linewidth=2, label='Inicio (Fase Base)')
        ax3.plot(t[idx_final] - 425, act[idx_final], color=datos_protocolo['color'], linewidth=2, linestyle='--', label='Final (Fase de Sintonía)')
        ax3.set_ylabel("Actividad", fontweight='bold', color='#2c3e50')
        ax3.set_title("3. Adaptación Dinámica: Transición Inicial vs. Estabilización", fontsize=10, fontweight='bold', color='#2c3e50')
        ax3.legend(loc='upper right', fontsize=8.5)
        ax3.grid(True, linestyle=':', alpha=0.6)

        # --- Gráfico 4 ---
        if fptd:
            sorted_fptd = np.sort(fptd)
            observed_cumulative = np.arange(1, len(sorted_fptd) + 1) / len(sorted_fptd) * 100
            tau_teorico = 80.0 if tipo_onda == 'sin' else (50.0 if tipo_onda == 'square' else 30.0)
            expected_cumulative = 100 / (1 + np.exp(-(sorted_fptd - latencia_activacion) / (tau_teorico * 0.3)))

            ax4.plot(sorted_fptd, expected_cumulative, color='#7f8c8d', linestyle='--', linewidth=2, label='Esperada')
            ax4.plot(sorted_fptd, observed_cumulative, color=datos_protocolo['color'], linewidth=2.5, label='Observada (Simulación)')
            ax4.fill_between(sorted_fptd, expected_cumulative, observed_cumulative, color=datos_protocolo['color'], alpha=0.15)
            ax4.axvline(latencia_activacion, color='#e74c3c', linestyle='-', linewidth=2, label=f'Umbral 50% ({latencia_activacion:.1f} ms)')

        ax4.set_ylabel("Activación (%)", fontweight='bold', color='#2c3e50')
        ax4.set_title("4. Evolución Temporal Local (Umbral de Activación Fenomenológica)", fontsize=10, fontweight='bold', color='#2c3e50')
        ax4.legend(loc='lower right', fontsize=8)
        ax4.grid(True, linestyle=':', alpha=0.6)
        ax4.set_xlim(0, 500)
        ax4.set_ylim(0, 105)

        # --- Gráfico 5 ---
        tiempo_sesion_max = exposicion_analitica_max * 1.8
        t_sesion = np.linspace(0, tiempo_sesion_max, 300)
        t_half = (exposicion_analitica_min + exposicion_analitica_max) / 2.0
        acumulacion_macro_neuroplastica = 100 * (1.0 - np.exp(-t_sesion / t_half)) * (1.0 - 0.15 * np.maximum(0, (t_sesion - exposicion_analitica_max) / exposicion_analitica_max))

        ax5.plot(t_sesion, acumulacion_macro_neuroplastica, color='#27ae60', linewidth=3, label='Integración Sostenida')
        ax5.axvspan(exposicion_analitica_min, exposicion_analitica_max, color=datos_protocolo['color'], alpha=0.25,
                    label=f'Ventana Óptima ({exposicion_analitica_min:.1f} - {exposicion_analitica_max:.1f} min)')

        ax5.set_xlabel("Tiempo de Escucha en Minutos (min)", fontweight='bold', color='#2c3e50')
        ax5.set_ylabel("Integración de Red (%)", fontweight='bold', color='#2c3e50')
        ax5.set_title("5. Acumulación Macro: Estabilización de la Plasticidad por Protocolo Dinámico", fontsize=10, fontweight='bold', color='#2c3e50')
        ax5.legend(loc='upper right', fontsize=8)
        ax5.grid(True, linestyle=':', alpha=0.6)
        ax5.set_xlim(0, tiempo_sesion_max)
        ax5.set_ylim(0, 110)

        plt.show()

    with output_guia:
        clear_output(wait=True)
        datos = configuraciones_ondas[selector_onda.value]
        comb = combinaciones_acusticas[dropdown_combinacion.value]
        print("======================================================================")
        print(f" MATRIZ DE DISEÑO ACÚSTICO: TMNE v2.5 (N = {dropdown_neuronas.value})")
        print("======================================================================")
        print(f"• Protocolo Base        : {selector_onda.value}")
        print(f"• Sistema Estimulado    : {datos['sistema']}")
        print(f"• Textura Seleccionada  : {dropdown_combinacion.value}")
        print(f"  -> Propiedad Textural : {comb['desc']}")
        print(f"• Impacto del Susurro   : {datos['susurro_impacto']}")
        print(f"• Objetivo Principal    : {datos['objetivo']}")
        print(f"• Ventajas en la Rutina : {datos['ventajas']}")
        print("======================================================================")

selector_onda.observe(actualizar_todo, names='value')
dropdown_forma_onda.observe(actualizar_todo, names='value')
dropdown_combinacion.observe(actualizar_todo, names='value')
dropdown_neuronas.observe(actualizar_todo, names='value')

display(widgets.VBox([
    widgets.HBox([selector_onda, dropdown_forma_onda]),
    widgets.HBox([dropdown_combinacion, dropdown_neuronas])
]))
display(output)
display(output_guia)
actualizar_todo(None)

Output()

Output()